# BIOINFORMATICS: INTRODUCTION TO PYTHON

## DAY 4 TRAINING PLAN

### Instructor: Neslihan Gokmen


### OVERVIEW

* Visualization (Seeing Beyond the Numbers with Matplotlib & Seaborn)
* Automation (Turning Code into a Factory with Functions)
* Closing Project (Generating an Automated QC Report and Pushing it to GitHub)

---

## Visualization - Seeing Beyond the Numbers

**Goal:** Turning the data we processed with Pandas and Numpy into professional plots (scatter, histogram, boxplot) that we can use in papers and presentations.

#### 1. Why Do We Plot Data?

The human brain cannot understand a table with millions of rows, or complex numbers (like p-values), in a few seconds. But we can notice a color or a trend line in milliseconds. In Python, we use two main libraries for plotting: **Matplotlib** (our basic brush) and **Seaborn** (our professional designer).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Activate the Seaborn theme so our plots look nicer and more scientific
sns.set_theme(style="whitegrid")

# Let's create some example RNA-Seq (gene expression) data
np.random.seed(42)
data = pd.DataFrame({
    'Gene': [f'Gene_{i}' for i in range(1, 101)],
    'Control_Expression': np.random.normal(50, 15, 100), # Control group average is 50
    'Cancer_Expression': np.random.normal(80, 25, 100),  # Cancer group is generally higher
    'Pathway': np.random.choice(['Apoptosis', 'Cell_Cycle', 'DNA_Repair'], 100)
})

print("Example Data Created (First 3 Rows):")
display(data.head(3))

#### 2. Histogram and Distribution Plots (Scatter Plot)

Let's plot our graphs to see the structure of our data.

In [ ]:
plt.figure(figsize=(12, 5))

# 1. HISTOGRAM: What is the expression distribution in the Cancer and Control groups?
plt.subplot(1, 2, 1) # 1 row, 2 columns, area 1
sns.histplot(data['Control_Expression'], color='blue', label='Control (Healthy)', kde=True, alpha=0.5)
sns.histplot(data['Cancer_Expression'], color='red', label='Cancer', kde=True, alpha=0.5)
plt.title('Gene Expression Distribution')
plt.xlabel('Expression Level')
plt.ylabel('Number of Genes')
plt.legend()



In [ ]:
# 2. SCATTER PLOT: What kind of relationship is there between the groups?
sns.scatterplot(data=data, x='Control_Expression', y='Cancer_Expression', hue='Pathway', palette='Set2')
plt.title('Control vs Cancer Expression')
plt.plot([0, 150], [0, 150], color='black', linestyle='--') # y=x reference line

plt.tight_layout()
plt.show()

3. Real Data

Now let's work with the Breast Cancer Wisconsin dataset, real data collected from hospitals that comes built into the Scikit-Learn machine learning library.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

# Activate the Seaborn theme so our plots look nicer and more scientific
sns.set_theme(style="whitegrid")

# 1. Load the real breast cancer dataset
cancer_data = load_breast_cancer()
df = pd.DataFrame(cancer_data.data, columns=cancer_data.feature_names)

# 2. Add the target (Diagnosis) column and convert the numbers to text
df['Diagnosis'] = cancer_data.target
df['Diagnosis'] = df['Diagnosis'].map({0: 'Malignant', 1: 'Benign'})

# 3. There are 30 columns in the dataset, let's pick the 4 most critical ones so we don't flood the screen
df = df[['mean radius', 'mean texture', 'mean area', 'mean smoothness', 'Diagnosis']]

# 4. Rename the columns
df.columns = ['Radius', 'Texture', 'Area', 'Smoothness', 'Diagnosis']

print("Wisconsin Breast Cancer Dataset (First 5 Patients):")
display(df.head())

4. Scatter Plot: Relationship Between Variables

Does the area increase as the tumor size (Radius) increases? After what size can it be considered risky?

In [ ]:
plt.figure(figsize=(8, 6))

# Scatter plot with Seaborn
# The hue='Diagnosis' parameter colors the patients differently based on their diagnosis
sns.scatterplot(data=df, x='Radius', y='Area', hue='Diagnosis', palette=['red', 'blue'], alpha=0.7)

plt.title('Relationship Between Tumor Radius and Area', fontsize=14)
plt.xlabel('Tumor Radius (mean)')
plt.ylabel('Tumor Area (mean)')

plt.show()

5. Boxplot and Violin Plot: Finding Outliers

The "Boxplot" is the plot type most commonly encountered in bioinformatics. It lets us see where the data is concentrated and find the 'abnormal' patients (outliers).

In [ ]:
plt.figure(figsize=(12, 5))

# 1. BOXPLOT
plt.subplot(1, 2, 1) # 1 row, 2 columns, area 1
sns.boxplot(data=df, x='Diagnosis', y='Radius', palette='pastel')
plt.title('Boxplot: Radius Distribution and Outliers (Dots)')

# 2. VIOLIN PLOT
plt.subplot(1, 2, 2) # 1 row, 2 columns, area 2
sns.violinplot(data=df, x='Diagnosis', y='Radius', palette='pastel', inner='quartile')
plt.title('Violin Plot: Shape of the Radius Density')

plt.tight_layout()
plt.show()

6. Heatmap: An X-Ray of Complex Tables

If we had 50 genes/features instead of 4, how would we find which ones move together (Correlation)? Instead of staring at huge tables, we use Heatmaps.

In [ ]:
plt.figure(figsize=(8, 6))

# The text column (Diagnosis) must be removed before calculating correlation
numeric_data = df.drop('Diagnosis', axis=1)

# Build the correlation matrix showing how similar each feature is to the others
correlation_matrix = numeric_data.corr()

# Draw the heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=1)

plt.title('Correlation Heatmap Between Features', fontsize=14)
plt.show()

7. Interactive 3D Plots (Plotly): The Star of Presentations

Seaborn and Matplotlib give great static (non-moving) images. But if we want to rotate the data, zoom in, and read a patient's values by hovering over their point during a presentation, we use the Plotly library.

In [ ]:
import plotly.express as px

# Interactive 3D Scatter Plot with Plotly
# We give 3 different features to the X, Y and Z axes
fig = px.scatter_3d(df, x='Radius', y='Texture', z='Area',
                    color='Diagnosis', # Color split by Diagnosis
                    color_discrete_map={'Malignant': 'red', 'Benign': 'blue'},
                    opacity=0.7,
                    title='Interactive 3D Tumor Analysis (Try rotating it with your mouse!)')

# Show the plot
fig.show()


---

## Automation - Turning Code into a Factory with Functions (`def`)

**Goal:** Turning code blocks we keep repeating (e.g. GC calculation) into a "Function" (our own custom command) to automate them.

#### 1. What is a Function?

On day 1 of the training we wrote 5-6 lines of code to calculate the GC ratio. So if we have 100 different genes, are we going to copy-paste that code 100 times? No! We'll put our code inside a **Function**.

Functions are **mini factories** that you feed raw material into, and that hand you back a processed product.

In [ ]:
# We invent our own custom command with the 'def' (define) keyword.
# Name: run_quality_control
# Raw material it wants (parameter): dna_sequence

def run_quality_control(dna_sequence):
    """This function analyzes the length and GC ratio of a DNA sequence."""

    sequence_upper = dna_sequence.upper()
    length = len(sequence_upper)

    g_count = sequence_upper.count('G')
    c_count = sequence_upper.count('C')
    gc_ratio = ((g_count + c_count) / length) * 100

    # Let's add a biological rule:
    if gc_ratio < 40 or gc_ratio > 60:
        status = "FAIL (Unbalanced GC)"
    else:
        status = "PASS"

    # The packaged product coming out of the factory (return)
    return {
        'Length': length,
        'GC_%': round(gc_ratio, 1),
        'QC_Status': status
    }

print("Function (factory) built successfully!")

#### 2. Using the Function in Automation

Now let's run hundreds of sequences through this factory with a single loop.

In [ ]:
# Say we have 3 different gene sequences from a database
gene_database = [
    "ATGCGTACGTAGCTAG", # Normal sequence
    "ATATATATATATATAT", # Only A and T (Problematic)
    "GCGCGCGCGCGCGCGC"  # Only G and C (Problematic)
]

analysis_results = []

# Automation starts: loop and function combined!
for i, gene in enumerate(gene_database):
    # Send each gene to our function (factory)
    result = run_quality_control(gene)
    analysis_results.append(result)

    print(f"Gene {i+1} Analysis: {result}")

3. Advanced Automation: Reverse Complement Finder

In DNA analyses we only read one strand of the sequence, and the computer has to figure out the other strand (the complementary strand) for us. Let's build a second factory for this, one that turns Adenine into Thymine, Guanine into Cytosine, and reverses the sequence.

In [ ]:
def find_reverse_complement(dna_sequence):
    """Calculates the reverse complement of the given DNA sequence."""
    # Biological base-pairing rules (using a Dictionary structure)
    base_pairs = {'A': 'T', 'T': 'A', 'G': 'C', 'C': 'G', 'N': 'N'}

    complement_sequence = ""
    dna_sequence = dna_sequence.upper()

    # Go through the letters one by one and find the complementary strand
    for base in dna_sequence:
        # If an unknown letter comes up, put '?'
        complement_sequence += base_pairs.get(base, '?')

    # Last step: reverse the text [::-1] to get the 5' -> 3' direction
    reverse_complement = complement_sequence[::-1]

    return reverse_complement

print("Function 2 (Reverse Complement Factory) built successfully!")

# Let's test it
example_dna = "ATGCGTA"
print(f"\nOriginal DNA: {example_dna}")
print(f"Reverse Complement: {find_reverse_complement(example_dna)}")

4. Big Data Automation: Pandas .apply()

If we have a table with thousands of rows, writing a "for" loop is inefficient and outdated. In Pandas tables, we can apply these factories (functions) we've built to an entire column in a single move with the .apply() command!

In [ ]:
import pandas as pd

# Say we have a table of gene sequences taken from 5 patients
gene_table = pd.DataFrame({
    'Patient_ID': ['P001', 'P002', 'P003', 'P004', 'P005'],
    'Sequence': ['ATGCGTAC', 'GCGCGCGC', 'ATATATAT', 'NNNN', 'TGGCCA']
})

print("--- RAW TABLE ---")
display(gene_table)



In [ ]:

print("\n--- AUTOMATION RUNNING (.apply) ---")
# 1. Calculate the reverse complements and write them to a new column
gene_table['Complementary_Strand'] = gene_table['Sequence'].apply(find_reverse_complement)

# 2. Apply our Quality Control (QC) factory and analyze the results
# (since our QC function returns a dictionary, we add pd.Series to spread it into separate columns)
qc_results = gene_table['Sequence'].apply(run_quality_control).apply(pd.Series)

# Combine the original table with the QC results (Concat)
final_report = pd.concat([gene_table, qc_results], axis=1)

display(final_report)

5. Real-World Scale Big Data Automation (Speed Test)

Everything worked nicely with 5 patients. But what happens if, like in the real world, we have 100,000 patient sequences? Let's run a speed test (benchmark) to see why the .apply() method replaces for loops.

In [ ]:
import numpy as np
import time

print("Building a huge dataset with 100,000 rows...")

# Generate 100,000 random DNA sequences (each 50 letters long)
big_table = pd.DataFrame({
    'Patient_ID': [f'P{i}' for i in range(1, 100001)],
    'Sequence': [''.join(np.random.choice(['A', 'T', 'G', 'C'], 50)) for _ in range(100000)]
})

print(f"Table ready! Row count: {big_table.shape[0]:,}\n")

print("Sending the huge dataset to the factory with .apply(). Please wait...")

# Start the stopwatch
start_time = time.time()

# Run quality control on all 100,000 rows with a single line of code
big_qc_results = big_table['Sequence'].apply(run_quality_control).apply(pd.Series)
big_report = pd.concat([big_table, big_qc_results], axis=1)

# Stop the stopwatch
end_time = time.time()
elapsed_time = end_time - start_time

print(f"DONE!")
print(f"Analyzing 100,000 patients took ONLY {elapsed_time:.2f} seconds!\n")

print("Here are the first 5 results:")
display(big_report.head())


---

## Closing Project - Generating a QC Report and Pushing it to GitHub

**Goal:** Producing a professional, statistical summary report (in Markdown format) of the huge dataset we analyzed. Then learning to use .gitignore, a bioinformatician's real secret, to filter out unnecessary large files and push only the code and report we wrote to GitHub.

#### 1. Automatic QC Summary Report Generation (Statistical Analysis)

We can't inspect a table with 100,000 rows in the real world. What's wanted is a nice report with statistics such as "How many patients passed? How many were left? What is our overall success rate?"

Bioinformaticians write reports in Markdown (.md) format, where tables look much nicer, instead of plain .txt. Let's generate the statistical report for 100,000 patients in seconds!

In [ ]:
import datetime

# Report generation time
timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M")

# Realistic QC Statistics Calculation
total_patients = len(big_report)
passed_patients = len(big_report[big_report['QC_Status'] == 'PASS'])
remaining_patients = total_patients - passed_patients
success_rate = (passed_patients / total_patients) * 100

# Prepare the report in Markdown (.md) format (bioinformatics standard)
report_text = f"""# BIOINFORMATICS QUALITY CONTROL (QC) REPORT
**Generated On:** {timestamp}
**Analyzed By:** Bioinformatics Python Team

## Analysis Summary
* **Total Patients Analyzed:** {total_patients:,}
* **Passed QC (PASS):** {passed_patients:,}
* **Failed QC (FAIL):** {remaining_patients:,}
* **Overall Success Rate:** {success_rate:.2f}%

## Review of Problematic Samples (First 5 FAIL Cases)
| Patient ID | GC Ratio | Status |
|---|---|---|
"""

# Pull a few examples of failed patients and add them to the Markdown table
failed_patients = big_report[big_report['QC_Status'] != 'PASS'].head(5)
for index, patient in failed_patients.iterrows():
    report_text += f"| {patient['Patient_ID']} | {patient['GC_%']}% | {patient['QC_Status']} |\n"

report_text += "\n*This report was generated automatically by Python automation.*\n"

# Save the file
with open("QC_Summary_Report_extended.md", "w", encoding="utf-8") as file:
    file.write(report_text)

print("QC_Summary_Report.md generated successfully! You can double-click it in the left panel to inspect it.")



## 1) Create a repo on GitHub
GitHub -> New repository -> name it `bioinformatics-qc` -> **add a README and .gitignore** -> Create repository

## 2) Doing everything in one go in Colab
In bioinformatics, code isn't saved in folders with names like "v1", "v2_final", "v3_final_final". Instead, a version control system called **Git** and a cloud repository called **GitHub** are used.

But there's a GOLDEN RULE: files larger than 100MB cannot be uploaded to GitHub. So we can't push huge data like DNA tables (VCF, FASTA) to GitHub!

*Let's see, using terminal commands (!git), how a repository is created (Init), how files are added (Add), and how changes are sealed (Commit):*

How do we tell Git "Ignore these files, only save the code and the report I wrote"? This is where the hidden file called .gitignore comes in.
Run the cell below **as a single block**:



In [ ]:
with open(".gitignore", "w") as f:
    f.write("""
*.fasta
*.vcf
*.csv
*.gz
sample_data/
.config/
""")

In [ ]:
# 2. GIT CONFIG (set your identity)


!git config --global user.name "your-github-username"
!git config --global user.email "your-email@example.com"

print("Git identity configured")



In [ ]:
# 3. GIT INIT (start the repo)

!git init
!git branch -m main   # modern branch name

print("Git repository initialized")

In [ ]:
# 4. CREATE AN EXAMPLE REPORT (Markdown)

report = """
# QC Analysis Report

## Summary
The data quality of 100,000 patients has been analyzed.

## Findings
- Missing data rate: 2.3%
- Average coverage: 35X
- Variant count after filtering: 1,234,567

## Conclusion
The data is suitable for downstream analysis.
"""

with open("QC_Summary_Report.md", "w") as f:
    f.write(report)

print("QC_Summary_Report.md created")



In [ ]:
# 5. GIT ADD (code + report only)

!git add .gitignore QC_Summary_Report.md QC_Summary_Report_extended.md

print("Files staged")

In [ ]:
# 6. COMMIT

!git commit -m "feat: ran QC analysis and generated an automatic Markdown report"

print("Commit complete")

Create a GitHub token

GitHub -> Settings

Developer settings

Personal access tokens

Fine-grained token

Repository access: the relevant repo or all repositories

Permission: Contents -> Read and Write

Generate token

Copy the token


### Pushing to GitHub - Making the Analysis Permanent

What we've done so far:

Created .gitignore
Initialized the Git repository
Committed the report

But there's an important point:

Right now all changes are only kept inside the Colab environment (temporary).

Everything is deleted when Colab closes.

In [ ]:
import getpass

token = getpass.getpass("Enter GitHub Token: ")

!git remote set-url origin https://{token}@github.com/your-username/bioinformatics-qc.git
!git push -u origin main